# Track B Anomaly Detection Evidence Notebook

## Purpose
This notebook presents governed Track B anomaly detection evidence for review and demo/report use. It is read-only, Colab-first, local-capable, fallback-aware, reproducibility-aware, leakage-aware, and aligned with the repository artifact structure and the Track B frontend bundle.

## Scope
- Track: Track B
- Task: anomaly detection
- Dataset identity: MVTec anomaly evidence from governed artifacts
- Canonical run: `b8ca43f5-0d53-4a42-ab37-b5fca9544a36`
- Canonical status: `production-canonical`
- Model: `autoencoder v0.1.0`

## What This Notebook Does
- Consumes existing governed JSON, checkpoint, metrics, metadata, inventory, post-hoc log, explainability artifacts, and the Track B frontend bundle.
- Presents anomaly score summaries, reconstruction loss, threshold behavior, metric cards, sample galleries, and traceability.
- Reports required and optional evidence availability clearly.

## What This Notebook Does Not Do
- It does not train, evaluate, regenerate, rewrite, or register artifacts.
- It does not replace `src/`, `configs/`, validation scripts, registries, or governed artifacts.
- It does not invent metrics, placeholder hashes, fake outputs, or silent success states.

This notebook is not the source of truth. The source of truth is governed artifacts, registries, and the Track B frontend bundle.


## 1. Runtime Detection
Detect whether the notebook is running in Colab or locally, report Python version, resolve the repository root, and show a safe device summary. This section does not start training or write files.

In [ ]:
from pathlib import Path
import json
import platform
import sys

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from IPython.display import display
except Exception:
    display = None

IS_COLAB = 'google.colab' in sys.modules
print(f'runtime={"colab" if IS_COLAB else "local"}')
print(f'python={platform.python_version()}')
print(f'platform={platform.platform()}')
print('device_summary=not queried by default; notebook is read-only evidence presentation')

## 2. Path And Artifact Resolution
The notebook assumes execution from the repository root when local. In Colab, mount or clone the repository first, then set `REPO_ROOT` below if auto-detection does not find the project. Required missing evidence fails clearly; optional missing evidence is reported.

In [ ]:
PROJECT_MARKERS = ['artifacts', 'scripts', 'notebooks']

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in PROJECT_MARKERS):
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd()).resolve()
RUN_ID = 'b8ca43f5-0d53-4a42-ab37-b5fca9544a36'
TRACK_ID = 'track_b'
TASK_TYPE = 'anomaly_detection'

def rel(path: str) -> Path:
    return REPO_ROOT / path

ARTIFACTS = [
    {'key': 'training_result', 'required': True, 'path': 'artifacts/models/analysis/training_results/training_result__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'checkpoint', 'required': True, 'path': 'artifacts/models/checkpoints/model_checkpoint__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.pt', 'kind': 'checkpoint'},
    {'key': 'anomaly_evaluation', 'required': True, 'path': 'artifacts/models/metrics/anomaly_detection_evaluation__b8ca43f5-0d53-4a42-ab37-b5fca9544a36__test.json', 'kind': 'json'},
    {'key': 'learning_curves', 'required': True, 'path': 'artifacts/models/metrics/anomaly_learning_curves__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'confusion_matrix', 'required': True, 'path': 'artifacts/models/metrics/anomaly_confusion_matrix__b8ca43f5-0d53-4a42-ab37-b5fca9544a36__train_test.json', 'kind': 'json'},
    {'key': 'artifact_inventory', 'required': True, 'path': 'artifacts/models/inventory/track_b_artifact_inventory__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'production_canonical_summary', 'required': True, 'path': 'artifacts/models/metadata/track_b_production_canonical_summary__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'full_production_canonical_summary', 'required': True, 'path': 'artifacts/models/metadata/track_b_full_production_canonical_summary__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'posthoc_log', 'required': True, 'path': 'artifacts/models/logs/track_b_posthoc_run_log__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'qualitative_samples', 'required': False, 'path': 'artifacts/models/explainability/b8ca43f5-0d53-4a42-ab37-b5fca9544a36/anomaly_qualitative_samples__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'reconstruction_explainability', 'required': False, 'path': 'artifacts/models/explainability/b8ca43f5-0d53-4a42-ab37-b5fca9544a36/anomaly_reconstruction_explainability__b8ca43f5-0d53-4a42-ab37-b5fca9544a36.json', 'kind': 'json'},
    {'key': 'frontend_anomaly_score_summary', 'required': True, 'path': 'artifacts/frontend/track_b/anomaly_score_summary.json', 'kind': 'json'},
    {'key': 'frontend_reconstruction_loss_summary', 'required': True, 'path': 'artifacts/frontend/track_b/reconstruction_loss_summary.json', 'kind': 'json'},
    {'key': 'frontend_threshold_behavior', 'required': True, 'path': 'artifacts/frontend/track_b/threshold_behavior.json', 'kind': 'json'},
    {'key': 'frontend_metric_cards', 'required': True, 'path': 'artifacts/frontend/track_b/metric_cards.json', 'kind': 'json'},
    {'key': 'frontend_sample_anomaly_gallery', 'required': True, 'path': 'artifacts/frontend/track_b/sample_anomaly_gallery.json', 'kind': 'json'},
    {'key': 'frontend_quality_decision_summary', 'required': True, 'path': 'artifacts/frontend/track_b/quality_decision_summary.json', 'kind': 'json'},
    {'key': 'frontend_frontend_anomaly_summary', 'required': True, 'path': 'artifacts/frontend/track_b/frontend_anomaly_summary.json', 'kind': 'json'},
    {'key': 'frontend_artifact_inventory', 'required': True, 'path': 'artifacts/frontend/track_b/artifact_inventory_frontend.json', 'kind': 'json'},
    {'key': 'frontend_roc_auc_pr_auc_summary', 'required': False, 'path': 'artifacts/frontend/track_b/roc_auc_pr_auc_summary.json', 'kind': 'json'},
]


## 3. Config, Dataset, And Track Summary
This section states the governed Track B identity used throughout the notebook. It is descriptive only; canonical values are loaded from governed artifacts in later sections.

In [ ]:
TRACK_SUMMARY = {
    'track_id': TRACK_ID,
    'task_type': TASK_TYPE,
    'dataset_id': 'mvtec_anomaly',
    'model_type': 'autoencoder',
    'model_version': '0.1.0',
    'run_id': RUN_ID,
    'run_config_id': 'autoencoder_train_v0_1_0',
    'evaluation_split': 'test',
    'canonical_status': 'production-canonical',
    'frontend_bundle_root': 'artifacts/frontend/track_b/',
    'frontend_bundle_status': 'validated',
    'retraining_needed_before_demo': False,
}
if pd:
    display(pd.DataFrame([TRACK_SUMMARY]))
else:
    print(json.dumps(TRACK_SUMMARY, indent=2))


## 4. Artifact Loading Helpers
Small helpers load JSON and report path status. They do not implement training, evaluation, registration, or canonical decision logic.

In [ ]:
def artifact_path(key: str) -> Path:
    return rel(ARTIFACT_BY_KEY[key]['path'])

def load_json_artifact(key: str, required: bool = True):
    item = ARTIFACT_BY_KEY[key]
    path = artifact_path(key)
    if not path.exists():
        message = f"missing {'required' if required else 'optional'} artifact: {key} -> {item['path']}"
        if required:
            raise FileNotFoundError(message)
        print(message)
        return None
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def assert_required_artifacts_present():
    missing = [item for item in ARTIFACTS if item['required'] and not rel(item['path']).exists()]
    if missing:
        details = '\n'.join(f"- {item['key']}: {item['path']}" for item in missing)
        raise FileNotFoundError(f'Missing required Track B evidence artifacts:\n{details}')
    print('required_artifacts_status=pass')

def show_table(rows):
    if pd:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(json.dumps(row, indent=2))

def nested_get(data, path, default=None):
    current = data
    for part in path:
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current

assert_required_artifacts_present()

## 5. Track B Evidence Inventory
Required artifacts must exist. Optional explainability and qualitative sample artifacts are reported honestly when unavailable.

In [ ]:
inventory_rows = []
for item in ARTIFACTS:
    path = rel(item['path'])
    inventory_rows.append({
        'artifact_key': item['key'],
        'required': item['required'],
        'exists': path.exists(),
        'kind': item['kind'],
        'size_bytes': path.stat().st_size if path.exists() else None,
        'path': item['path'],
    })
show_table(inventory_rows)

## 6. Training Result Summary
Load the governed TrainingResult and post-hoc log. This section reports run identity, model type, dataset, config, status, canonical status, and available epoch/loss evidence without rerunning training.

In [ ]:
training_result = load_json_artifact('training_result')
posthoc_log = load_json_artifact('posthoc_log')

identity = training_result.get('identity', {})
metadata = training_result.get('metadata', {})
metrics = training_result.get('metrics', {})
training_rows = [{
    'run_id': identity.get('run_id'),
    'track_id': metadata.get('track_id') or posthoc_log.get('track_id'),
    'task_type': identity.get('task_type') or metadata.get('task_type_from_loader'),
    'model_type': identity.get('model_type') or metadata.get('model_type'),
    'dataset_id': metadata.get('dataset_id') or posthoc_log.get('dataset_id'),
    'config_id': identity.get('run_config_id') or metadata.get('training_config_id'),
    'is_experiment': identity.get('is_experiment'),
    'run_status': posthoc_log.get('run_status'),
    'canonical_status': metadata.get('canonical_status') or posthoc_log.get('canonical_status'),
    'epochs': metadata.get('epochs') or metadata.get('real_training_epoch_count'),
    'batches_per_epoch': metadata.get('real_training_batches_per_epoch'),
    'batch_size': metadata.get('real_training_batch_size'),
    'reconstruction_loss': metrics.get('reconstruction_loss'),
    'original_runtime_log_available': posthoc_log.get('original_runtime_log_available'),
    'log_generation_mode': posthoc_log.get('log_generation_mode'),
}]
show_table(training_rows)
print('posthoc_note=Original runtime log availability is reported from the governed post-hoc log; the notebook does not infer runtime history.')

## 7. Anomaly Evaluation Metrics
Load the governed anomaly evaluation summary and show available metrics, threshold, score summaries, and counts. Missing metrics are left blank rather than invented.

In [ ]:
evaluation = load_json_artifact('anomaly_evaluation')
eval_metrics = evaluation.get('metrics', {})
eval_counts = evaluation.get('counts', {})
metric_row = {
    'roc_auc': eval_metrics.get('roc_auc'),
    'precision': eval_metrics.get('precision'),
    'recall': eval_metrics.get('recall'),
    'f1': eval_metrics.get('f1'),
    'threshold_strategy': evaluation.get('threshold_strategy'),
    'threshold': evaluation.get('threshold'),
    'train_score_count': eval_counts.get('train_score_count'),
    'test_score_count': eval_counts.get('test_score_count'),
    'normal_test_count': eval_counts.get('normal_test_count'),
    'anomaly_test_count': eval_counts.get('anomaly_test_count'),
    'correct_count': eval_counts.get('correct_count'),
    'incorrect_count': eval_counts.get('incorrect_count'),
}
show_table([metric_row])
print('score_definition=' + str(evaluation.get('score_definition')))

## 8. Learning Curves
Render train and validation loss curves when the governed curve schema supports plotting. If a curve is empty or too short, the notebook reports the available structure.

In [ ]:
learning_curves = load_json_artifact('learning_curves')
curves = learning_curves.get('curves', {}) if isinstance(learning_curves, dict) else {}
curve_rows = [{'curve': name, 'points': len(values), 'values': values} for name, values in curves.items() if isinstance(values, list)]
show_table(curve_rows)

if plt and any(row['points'] > 0 for row in curve_rows):
    plt.figure(figsize=(7, 4))
    for name, values in curves.items():
        if isinstance(values, list) and values:
            plt.plot(range(1, len(values) + 1), values, marker='o', label=name)
    plt.title('Track B Learning Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('learning_curve_plot_status=unavailable_or_empty')

## 9. Confusion Matrix / Threshold Summary
Load the governed anomaly confusion matrix. The threshold separates normal from anomaly predictions based on reconstruction error; values above the threshold are predicted anomaly.

In [ ]:
confusion = load_json_artifact('confusion_matrix')
print(f"threshold={confusion.get('threshold')}")
split_rows = []
for split_name, split_data in confusion.get('splits', {}).items():
    counts = split_data.get('counts', {})
    split_rows.append({'split': split_name, **counts})
show_table(split_rows)

if plt:
    splits = confusion.get('splits', {})
    for split_name, split_data in splits.items():
        matrix = split_data.get('matrix')
        labels = split_data.get('labels', [])
        if not matrix or not all(isinstance(row, list) for row in matrix):
            print(f'confusion_matrix_plot_status={split_name}:unavailable_shape')
            continue
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(matrix, cmap='Blues')
        ax.set_title(f'{split_name.title()} Confusion Matrix')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)
        for i, row in enumerate(matrix):
            for j, value in enumerate(row):
                ax.text(j, i, str(value), ha='center', va='center', color='black')
        plt.tight_layout()
        plt.show()
else:
    print('confusion_matrix_plot_status=matplotlib_unavailable')

## 10. Qualitative Samples
Load optional qualitative samples when available. Missing optional image files are reported rather than treated as evidence.

In [ ]:
qualitative = load_json_artifact('qualitative_samples', required=False)
qual_rows = []
if qualitative:
    for sample in qualitative.get('samples', [])[:12]:
        row = {
            'sample_id': sample.get('sample_id'),
            'image_path': sample.get('image_path'),
            'true_label': sample.get('true_label'),
            'predicted_label': sample.get('predicted_label'),
            'anomaly_score': sample.get('anomaly_score'),
            'correct': sample.get('correct'),
            'input_exists': rel(sample.get('input_path', '')).exists() if sample.get('input_path') else None,
            'reconstruction_exists': rel(sample.get('reconstruction_path', '')).exists() if sample.get('reconstruction_path') else None,
            'heatmap_exists': rel(sample.get('heatmap_path', '')).exists() if sample.get('heatmap_path') else None,
            'overlay_exists': rel(sample.get('overlay_path', '')).exists() if sample.get('overlay_path') else None,
        }
        qual_rows.append(row)
show_table(qual_rows if qual_rows else [{'status': 'optional qualitative samples unavailable'}])

## 11. Explainability
Reconstruction explainability compares the input image to its reconstruction. Heatmaps and overlays visualize reconstruction error regions. These are presentation artifacts only and are not generated by this notebook.

In [ ]:
explainability = load_json_artifact('reconstruction_explainability', required=False)
explain_rows = []
if explainability:
    for sample in explainability.get('heatmaps', [])[:6]:
        explain_rows.append({
            'sample_id': sample.get('sample_id'),
            'method': sample.get('method') or explainability.get('method'),
            'true_label': sample.get('true_label'),
            'predicted_label': sample.get('predicted_label'),
            'anomaly_score': sample.get('anomaly_score'),
            'input_path': sample.get('input_path'),
            'reconstruction_path': sample.get('reconstruction_path'),
            'heatmap_path': sample.get('heatmap_path'),
            'overlay_path': sample.get('overlay_path'),
            'overlay_exists': rel(sample.get('overlay_path', '')).exists() if sample.get('overlay_path') else None,
        })
show_table(explain_rows if explain_rows else [{'status': 'optional explainability unavailable'}])

if plt and explainability:
    try:
        import matplotlib.image as mpimg
        display_count = 0
        for sample in explainability.get('heatmaps', [])[:3]:
            paths = [sample.get('input_path'), sample.get('reconstruction_path'), sample.get('heatmap_path'), sample.get('overlay_path')]
            existing = [rel(path) for path in paths if path and rel(path).exists()]
            if not existing:
                continue
            fig, axes = plt.subplots(1, len(existing), figsize=(3 * len(existing), 3))
            if len(existing) == 1:
                axes = [axes]
            for ax, image_path in zip(axes, existing):
                ax.imshow(mpimg.imread(image_path))
                ax.set_title(image_path.name)
                ax.axis('off')
            plt.tight_layout()
            plt.show()
            display_count += 1
        print(f'explainability_image_sets_displayed={display_count}')
    except Exception as exc:
        print(f'explainability_image_display_status=unavailable: {exc}')
else:
    print('explainability_image_display_status=matplotlib_or_metadata_unavailable')

## 12. Frontend-Ready Artifact Summary
The Track B frontend bundle is the presentation layer for the canonical run. It is the primary demo/report surface and should be read instead of rebuilding metrics. PR AUC is unavailable in the governed evidence, so the bundle records that absence honestly instead of fabricating a value.


In [ ]:
frontend_bundle_keys = [
    'frontend_anomaly_score_summary',
    'frontend_reconstruction_loss_summary',
    'frontend_threshold_behavior',
    'frontend_metric_cards',
    'frontend_sample_anomaly_gallery',
    'frontend_quality_decision_summary',
    'frontend_frontend_anomaly_summary',
    'frontend_artifact_inventory',
    'frontend_roc_auc_pr_auc_summary',
]

frontend_rows = []
for key in frontend_bundle_keys:
    item = ARTIFACT_BY_KEY[key]
    payload = load_json_artifact(key, required=item['required'])
    row = {
        'artifact_key': key,
        'required': item['required'],
        'exists': rel(item['path']).exists(),
        'path': item['path'],
    }
    if payload:
        row['artifact_type'] = payload.get('artifact_type')
        row['run_id'] = payload.get('run_id')
        row['summary'] = payload.get('summary') or payload.get('safe_interpretation') or payload.get('gallery_explanation') or payload.get('safe_demo_wording') or payload.get('plain_language_explanation')
        if key == 'frontend_anomaly_score_summary':
            row['roc_auc'] = nested_get(payload, ['normal_vs_anomaly_score_separation', 'roc_auc'])
            row['test_mean_score'] = nested_get(payload, ['anomaly_score_statistics', 'test', 'mean'])
        elif key == 'frontend_reconstruction_loss_summary':
            row['reconstruction_loss'] = payload.get('final_reconstruction_loss')
        elif key == 'frontend_threshold_behavior':
            row['threshold'] = payload.get('selected_threshold')
            row['precision'] = nested_get(payload, ['selected_threshold_metrics', 'precision'])
            row['recall'] = nested_get(payload, ['selected_threshold_metrics', 'recall'])
            row['f1'] = nested_get(payload, ['selected_threshold_metrics', 'f1'])
        elif key == 'frontend_metric_cards':
            row['selected_model'] = payload.get('selected_model_name')
            row['recommended_threshold'] = payload.get('recommended_threshold')
        elif key == 'frontend_sample_anomaly_gallery':
            row['gallery_sample_count'] = payload.get('gallery_sample_count')
            row['counts_by_error_type'] = payload.get('counts_by_error_type')
        elif key == 'frontend_quality_decision_summary':
            row['canonical_status'] = payload.get('canonical_status')
            row['quality_target_status'] = payload.get('quality_target_status')
            row['production_ready'] = payload.get('production_ready')
            row['deployment_candidate'] = payload.get('deployment_candidate')
        elif key == 'frontend_frontend_anomaly_summary':
            row['key_metrics'] = payload.get('key_metrics')
        elif key == 'frontend_artifact_inventory':
            row['bundle_artifact_count'] = payload.get('bundle_artifact_count')
            row['source_artifact_count'] = payload.get('source_artifact_count')
            row['missing_optional_files'] = payload.get('missing_optional_files')
        elif key == 'frontend_roc_auc_pr_auc_summary':
            row['status'] = payload.get('status')
    else:
        row['status'] = 'unavailable_in_governed_evidence'
    frontend_rows.append(row)

show_table(frontend_rows)
print('frontend_bundle_root=artifacts/frontend/track_b/')
print('pr_auc_status=unavailable_in_governed_evidence')


## 13. Quality / Canonical Decision Summary
Summarize the canonical Track B decision from the governed frontend bundle. This is a presentation layer for review and demo use; it does not rewrite or recompute governed evidence.


In [ ]:
quality_summary = load_json_artifact('frontend_quality_decision_summary')
frontend_summary = load_json_artifact('frontend_frontend_anomaly_summary')
bundle_inventory = load_json_artifact('frontend_artifact_inventory')

quality_rows = [{
    'run_id': quality_summary.get('run_id'),
    'model_type': quality_summary.get('model_type'),
    'model_version': quality_summary.get('model_version'),
    'canonical_status': quality_summary.get('canonical_status'),
    'quality_target_status': quality_summary.get('quality_target_status'),
    'production_ready': quality_summary.get('production_ready'),
    'deployment_candidate': quality_summary.get('deployment_candidate'),
    'recommendation_status': quality_summary.get('recommendation_status'),
    'threshold': quality_summary.get('threshold'),
    'safe_wording': quality_summary.get('safe_wording'),
}]
show_table(quality_rows)
show_table([{
    'summary_type': frontend_summary.get('artifact_type'),
    'summary': frontend_summary.get('summary'),
    'roc_auc': nested_get(frontend_summary, ['key_metrics', 'roc_auc']),
    'precision': nested_get(frontend_summary, ['key_metrics', 'precision']),
    'recall': nested_get(frontend_summary, ['key_metrics', 'recall']),
    'f1': nested_get(frontend_summary, ['key_metrics', 'f1']),
}])
show_table([{
    'bundle_artifact_count': bundle_inventory.get('bundle_artifact_count'),
    'source_artifact_count': bundle_inventory.get('source_artifact_count'),
    'missing_optional_files': bundle_inventory.get('missing_optional_files'),
    'safe_demo_wording': bundle_inventory.get('safe_demo_wording'),
}])


## 14. Limitations
- This notebook is an evidence/presentation layer, not canonical training or evaluation logic.
- No training, evaluation builders, registry updates, or timestamp rewrites are executed here.
- Outputs are based on existing governed artifacts and the Track B frontend bundle only.
- The governed evidence does not provide PR AUC, so this notebook must state that it is unavailable rather than fabricating it.
- Missing optional qualitative or explainability files are reported honestly and must not be replaced with fake outputs.
- Threshold-based anomaly classification should be reviewed with domain expectations because false negatives and false positives carry operational risk.
- This notebook does not claim fully production-ready or deployment-safe status.


## 15. Final Decision Summary
Summarize governed Track B status from existing evidence and the Track B frontend bundle. This final section reports the committed evidence state and the demo/report-facing summary only.


In [ ]:
frontend_quality_summary = load_json_artifact('frontend_quality_decision_summary')
frontend_anomaly_summary = load_json_artifact('frontend_frontend_anomaly_summary')
frontend_inventory = load_json_artifact('frontend_artifact_inventory')

final_row = {
    'track_b_governance_evidence_status': 'pass' if frontend_quality_summary.get('canonical_status') == 'production-canonical' else 'review_required',
    'canonical_status': frontend_quality_summary.get('canonical_status'),
    'run_id': frontend_quality_summary.get('run_id'),
    'model_type': frontend_quality_summary.get('model_type'),
    'model_version': frontend_quality_summary.get('model_version'),
    'quality_target_status': frontend_quality_summary.get('quality_target_status'),
    'production_ready': frontend_quality_summary.get('production_ready'),
    'deployment_candidate': frontend_quality_summary.get('deployment_candidate'),
    'recommended_threshold': frontend_quality_summary.get('threshold'),
    'roc_auc': nested_get(frontend_anomaly_summary, ['key_metrics', 'roc_auc']),
    'precision': nested_get(frontend_anomaly_summary, ['key_metrics', 'precision']),
    'recall': nested_get(frontend_anomaly_summary, ['key_metrics', 'recall']),
    'f1': nested_get(frontend_anomaly_summary, ['key_metrics', 'f1']),
    'pr_auc_status': 'unavailable_in_governed_evidence',
    'frontend_bundle_path': frontend_inventory.get('bundle_directory'),
    'next_step': frontend_quality_summary.get('next_recommended_step'),
}
show_table([final_row])
print('safe_wording=' + str(frontend_quality_summary.get('safe_wording')))
print('forbidden_wording=' + json.dumps(frontend_quality_summary.get('forbidden_wording', [])))
print('limitations=' + json.dumps(frontend_quality_summary.get('limitations', [])))
